## Importing Libraries

In [107]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

In [108]:
!gdown 1POmzYEQZgZVqJO3F_cyNcd4QmoNE_mCv

Downloading...
From: https://drive.google.com/uc?id=1POmzYEQZgZVqJO3F_cyNcd4QmoNE_mCv
To: /content/bbc-news-data.csv
100% 5.08M/5.08M [00:00<00:00, 41.5MB/s]


## Loading the dataset

In [109]:
df = pd.read_csv("/content/bbc-news-data.csv", sep="\t")
df

,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...
...,...,...,...,...
2220,tech,397.txt,BT program to beat dialler scams,BT is introducing two initiatives to help bea...
2221,tech,398.txt,Spam e-mails tempt net shoppers,Computer users across the world continue to i...
2222,tech,399.txt,Be careful how you code,A new European directive could put software w...
2223,tech,400.txt,US cyber security chief resigns,The man making sure US computer networks are ...


In [110]:
# Dropping the empty sub-title
print(df.shape)
df = df[((~df.title.isnull()) & (~df.content.isnull()))].reset_index(drop=True)
print(df.shape)

df

(2225, 4)
(2225, 4)


,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...
...,...,...,...,...
2220,tech,397.txt,BT program to beat dialler scams,BT is introducing two initiatives to help bea...
2221,tech,398.txt,Spam e-mails tempt net shoppers,Computer users across the world continue to i...
2222,tech,399.txt,Be careful how you code,A new European directive could put software w...
2223,tech,400.txt,US cyber security chief resigns,The man making sure US computer networks are ...


## Data Cleaning

In [111]:
def data_cleaning(text):

    # Lower the words in the sentence
    cleaned = text.lower()

    # Replace the full stop with a full stop and space
    cleaned = cleaned.replace(".", ". ")

    # Remove the stop words : optional pre-processing step
    tokens = [word for word in cleaned.split() if not word in stop_words]

    # Remove the punctuations
    tokens = [tok.translate(str.maketrans(' ', ' ', string.punctuation)) for tok in tokens]

    # Joining the tokens back to form the sentence
    cleaned = " ".join(tokens)

    # Remove any extra spaces
    cleaned = cleaned.strip()

    return cleaned


In [112]:
df['clean_title'] = df['title'].apply(data_cleaning)

## Train-Test Split

In [113]:
train_X, test_X, train_Y, test_Y = train_test_split(
    df['clean_title'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']   # important for balanced classes
)

train_X = train_X.reset_index(drop=True)
test_X  = test_X.reset_index(drop=True)
train_Y = train_Y.reset_index(drop=True)
test_Y  = test_Y.reset_index(drop=True)

## Encode Labels

In [114]:
label_encoder = LabelEncoder()

train_Y_enc = label_encoder.fit_transform(train_Y)
test_Y_enc  = label_encoder.transform(test_Y)

num_classes = len(label_encoder.classes_)
print("Classes:", label_encoder.classes_)

Classes: ['business' 'entertainment' 'politics' 'sport' 'tech']


## Tokenization

In [115]:
max_vocab = 5000

tokenizer = Tokenizer(num_words=max_vocab, oov_token="<OOV>")
tokenizer.fit_on_texts(train_X)

train_seq = tokenizer.texts_to_sequences(train_X)
test_seq  = tokenizer.texts_to_sequences(test_X)

## Padding

In [116]:
max_len = 30   # titles are short

train_pad = pad_sequences(train_seq, maxlen=max_len, padding='post')
test_pad  = pad_sequences(test_seq,  maxlen=max_len, padding='post')

## RNN

### Build Model

In [117]:
embedding_dim = 64
rnn_units = 64

model = Sequential([
    Input(shape=(max_len,)),   # ✅ fixes unbuilt issue
    Embedding(input_dim=max_vocab, output_dim=embedding_dim, mask_zero=True),
    SimpleRNN(rnn_units, activation='tanh'),
    Dense(num_classes, activation='softmax')
])

model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ (None, 30, 64)         │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_9 (SimpleRNN)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 328,581 (1.25 MB)

 Trainable params: 328,581 (1.25 MB)

 Non-trainable params: 0 (0.00 B)

### Compile

In [118]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # ✅ since labels are integers
    metrics=['accuracy']
)

### Train

In [119]:
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_pad, train_Y_enc,
    validation_data=(test_pad, test_Y_enc),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.3444 - loss: 1.5443 - val_accuracy: 0.4337 - val_loss: 1.3895
Epoch 2/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8022 - loss: 0.7924 - val_accuracy: 0.6697 - val_loss: 0.8834
Epoch 3/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9803 - loss: 0.1431 - val_accuracy: 0.7079 - val_loss: 0.8755
Epoch 4/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9989 - loss: 0.0249 - val_accuracy: 0.7169 - val_loss: 0.9440
Epoch 5/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 1.0000 - loss: 0.0100 - val_accuracy: 0.7124 - val_loss: 1.0308
Epoch 6/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 1.0000 - loss: 0.0056 - val_accuracy: 0.7146 - val_loss: 1.0867
Epoch 7/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 1.0000 - loss: 0.0039 - val_accuracy: 0.7101 - val_loss: 1.1326


### Evaluate

In [120]:
loss, acc = model.evaluate(test_pad, test_Y_enc)
print(f"Test Accuracy: {acc:.4f}")

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7169 - loss: 0.9440 
Test Accuracy: 0.7169


### Predict on New Text

In [121]:
def predict_text(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_len, padding='post')

    pred = model.predict(pad)
    label = label_encoder.inverse_transform([np.argmax(pred)])

    return label[0]

print(predict_text("Stock market sees huge growth"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
business


In [122]:
print(predict_text("New AI model beats previous benchmarks"))
# 'tech'

print(predict_text("Cybersecurity threats increase worldwide"))
# 'tech'

print(predict_text("Tech company launches new smartphone"))
# 'tech'

print(predict_text("Software update improves performance"))
# 'tech'

print(predict_text("Government passes new economic policy"))
# 'politics'

print(predict_text("Election results spark debate nationwide"))
# 'politics'

print(predict_text("Prime minister addresses the nation"))
# 'politics'

print(predict_text("Team wins championship after thrilling final"))
# 'sport'

print(predict_text("Player scores hat-trick in match"))
# 'sport'

print(predict_text("Olympics preparations underway"))
# 'sport'

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
tech
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
sport
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
business
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
tech
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
entertainment
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
politics
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
business
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
entertainment
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
tech
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
sport


## LSTM

In [123]:
embedding_dim = 64
lstm_units = 64

model_lstm = Sequential([
    Input(shape=(max_len,)),
    Embedding(input_dim=max_vocab, output_dim=embedding_dim, mask_zero=True),

    LSTM(lstm_units, return_sequences=False),   # 🔥 key change

    Dense(64, activation='relu'),               # optional extra layer
    Dense(num_classes, activation='softmax')
])

model_lstm.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_10 (Embedding)        │ (None, 30, 64)         │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 357,509 (1.36 MB)

 Trainable params: 357,509 (1.36 MB)

 Non-trainable params: 0 (0.00 B)

In [125]:
model_lstm.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [126]:
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)

history = model_lstm.fit(
    train_pad, train_Y_enc,
    validation_data=(test_pad, test_Y_enc),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.3539 - loss: 1.5920 - val_accuracy: 0.5011 - val_loss: 1.5209
Epoch 2/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7713 - loss: 0.9017 - val_accuracy: 0.8067 - val_loss: 0.6734
Epoch 3/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9713 - loss: 0.1366 - val_accuracy: 0.8225 - val_loss: 0.6933
Epoch 4/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9966 - loss: 0.0254 - val_accuracy: 0.8180 - val_loss: 0.7258
Epoch 5/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 1.0000 - loss: 0.0061 - val_accuracy: 0.8180 - val_loss: 0.7805
Epoch 6/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 1.0000 - loss: 0.0028 - val_accuracy: 0.8202 - val_loss: 0.8300


In [128]:
loss, acc = model_lstm.evaluate(test_pad, test_Y_enc)
print("Test Accuracy:", acc)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8225 - loss: 0.6933
Test Accuracy: 0.8224719166755676


In [131]:
def predict_text(text):
    text = data_cleaning(text)
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_len, padding='post')

    pred = model.predict(pad)
    label = label_encoder.inverse_transform([np.argmax(pred)])

    return label[0]

In [132]:
print(predict_text("Stock market sees huge growth"))
print(predict_text("New AI system beats competitors"))
print(predict_text("Team wins championship final"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step
business
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
tech
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
entertainment
